# 🧩 Exercícios — Bellman-Ford

Soluções práticas para explorar o algoritmo de Bellman-Ford: parada antecipada, validação, geração de instâncias, detecção de ciclos negativos e comparação com Dijkstra quando aplicável.

In [ ]:
from math import inf
from typing import Any, Dict, Tuple, List, Optional
import random
import networkx as nx

def bellman_ford_early_stop(G: nx.DiGraph, s: Any, weight: str = 'weight'):
    d = {v: inf for v in G.nodes()}
    pi = {v: None for v in G.nodes()}
    d[s] = 0.0
    V = list(G.nodes())
    for _ in range(len(V)-1):
        updated = False
        for u, v, data in G.edges(data=True):
            w = float(data.get(weight, 1.0))
            if d[u] + w < d[v]:
                d[v] = d[u] + w
                pi[v] = u
                updated = True
        if not updated:
            break
    neg_cycle = False
    for u, v, data in G.edges(data=True):
        w = float(data.get(weight, 1.0))
        if d[u] + w < d[v]:
            neg_cycle = True
            break
    return d, pi, neg_cycle

def generate_random_digraph(n=10, m=20, wmin=-5, wmax=10, seed=42, avoid_negative_cycles=True):
    random.seed(seed)
    D = nx.DiGraph()
    D.add_nodes_from(range(n))
    edges = set()
    while len(edges) < m:
        u = random.randrange(n)
        v = random.randrange(n)
        if u != v:
            edges.add((u,v))
    for u,v in edges:
        D.add_edge(u, v, weight=random.uniform(wmin, wmax))
    if avoid_negative_cycles:
        # Ajuste opcional: garante não haver ciclo negativo simples 2-ciclo u<->v
        for u,v in list(D.edges()):
            if D.has_edge(v,u):
                w1 = D[u][v]['weight']
                w2 = D[v][u]['weight']
                if w1 + w2 < 0:
                    D[u][v]['weight'] = abs(w1)
                    D[v][u]['weight'] = abs(w2)
    return D

## 1) Parada antecipada (early stopping)

In [ ]:
D = generate_random_digraph(n=20, m=40, seed=1)
d, pi, neg = bellman_ford_early_stop(D, 0)
print('Ciclo negativo?', neg)
print({k: round(v,2) for k,v in d.items() if v < inf})

## 2) Comparar com NetworkX (quando suportado)

In [ ]:
try:
    from networkx.algorithms.shortest_paths.weighted import single_source_bellman_ford
    dist_nx, pred_nx = single_source_bellman_ford(D, 0)
    print('Distâncias NX (aprox.):', {k: round(v,2) for k,v in dist_nx.items()})
except Exception as e:
    print('NetworkX single_source_bellman_ford indisponível:', e)

## 3) Detectar e exibir um ciclo negativo

In [ ]:
H = nx.DiGraph()
H.add_weighted_edges_from([(0,1,1),(1,2,-2),(2,0,-2),(2,3,3)])
d2, pi2, neg2 = bellman_ford_early_stop(H, 0)
print('Tem ciclo negativo?', neg2)
# Para ilustrar o ciclo, reusa-se a rotina do notebook principal se desejar